In [8]:
run_id = 1782873335
env = 'dev'

In [9]:
from databricks.connect.session import DatabricksSession as SparkSession
import pyspark.sql.functions as f
from pyspark.sql.window import *
from datetime import datetime, timedelta
import yaml

with open('/home/ken/projects/box-office/src/box_office/config.yaml', 'r') as file:
    config = yaml.safe_load(file).get(env)
catalog = config.get('catalog')
franchises_config = config.get('seatmaps')
snapshot_folderpath = f'{config.get('seatmaps').get('snapshot_folderpath').format(run_id=run_id)}'

In [10]:
spark = SparkSession.builder.profile("main").serverless().getOrCreate()

In [11]:
seatmaps = spark.read.json(snapshot_folderpath)

In [ ]:
# seatmaps.printSchema()
# seatmaps.head(5)

root
 |-- _corrupt_record: string (nullable = true)
 |-- areas: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- availableSeatCount: long (nullable = true)
 |    |    |-- code: string (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- isReservedSeating: boolean (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- ticketInfo: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- admissionsPerTicket: long (nullable = true)
 |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |-- desc: string (nullable = true)
 |    |    |    |    |-- fee: string (nullable = true)
 |    |    |    |    |-- price: string (nullable = true)
 |    |    |    |    |-- reservedSeating: boolean (nullable = true)
 |    |    |-- totalSeatCount: long (nullable = true)
 |-- auditoriumId: long (nullable = true)
 |-- backgroundHeight: double (nullable = tr

[Row(_corrupt_record='<!DOCTYPE html>', areas=None, auditoriumId=None, backgroundHeight=None, backgroundSvg=None, backgroundWidth=None, chainCode=None, id=None, infoBoxes=None, mapOffsetX=None, mapOffsetY=None, maxTicketLimit=None, seatBlocks=None, seats=None, showtimeId=None, surcharges=None, theaterId=None, theaterName=None, tmsId=None, totalAvailableSeatCount=None, totalHeight=None, totalSeatCount=None, totalWidth=None),
 Row(_corrupt_record='<!--[if IE 8]>         <html class="no-js lt-ie9" lang="en" > <![endif]-->', areas=None, auditoriumId=None, backgroundHeight=None, backgroundSvg=None, backgroundWidth=None, chainCode=None, id=None, infoBoxes=None, mapOffsetX=None, mapOffsetY=None, maxTicketLimit=None, seatBlocks=None, seats=None, showtimeId=None, surcharges=None, theaterId=None, theaterName=None, tmsId=None, totalAvailableSeatCount=None, totalHeight=None, totalSeatCount=None, totalWidth=None),
 Row(_corrupt_record='<!--[if gt IE 8]><!--> <html class="no-js" lang="en" > <!--<![e

# ticketInfo (load)

In [13]:
from pyspark.sql.window import Window
ticketInfo_new = (seatmaps
    .filter(f.col('areas').isNotNull())
    .withColumn('areas', f.explode('areas'))
    .select('showtimeId', f.explode('areas.ticketInfo').alias('ticketInfo'))
    .select('showtimeId', 'ticketInfo.*')
    .withColumn('rn', f.row_number().over(Window.partitionBy('showtimeId', 'code').orderBy(f.lit(1).desc())))
    .filter(f.col('rn') == 1)
    .drop('rn')    
)

# ticketInfo.show()

In [ ]:
target_fqn = f'{catalog}.base.ticketInfo'
ticketInfo_new.limit(0).write.mode('ignore').saveAsTable(target_fqn)

(
    ticketInfo_new
    .mergeInto(target_fqn, (ticketInfo_new.showtimeId==f.col('ticketInfo.showtimeId')) & (ticketInfo_new.code== f.col('ticketInfo.code')))
        .whenMatched().updateAll()
        .whenNotMatched().insertAll()
        .withSchemaEvolution() 
        .merge()

)

# seatReservationHistory (load)

In [15]:
"""
    seatReservationHistory - Historical log of seat situation per snapshot
    Unique on:
        showtimeId, snapshot_ts
"""

seatReservationHistory_new = (
    seatmaps
    .withColumns({
        'run_id': f.lit(run_id),
        'snapshot_ts': f.col('snapshot_ts') if 'snapshot_ts' in seatmaps.columns else f.lit(run_id)
    })
    .drop('areas', 'backgroundSvg')
    .selectExpr('showtimeId', 'snapshot_ts', '* except(showtimeId, snapshot_ts)') # reordering columns for sanity sake 
    .withColumn('rn', f.expr('row_number() over(partition by showtimeId, snapshot_ts order by 1)')) # noticing dupes. Maybe duplicate requests? Idk but deduping either way
    .filter(f.col('rn') == 1)
)

# seatReservationHistory_new.show()
seatReservationHistory_new.count()

192

In [ ]:
target_fqn=f'{catalog}.base.seatReservationHistory'

(
    seatReservationHistory_new
    .mergeInto(target_fqn, (seatReservationHistory_new.showtimeId==f.col('seatReservationHistory.showtimeId')) & (seatReservationHistory_new.snapshot_ts==f.col('seatReservationHistory.snapshot_ts')))
        .whenMatched().updateAll()
        .whenNotMatched().insertAll()
        .withSchemaEvolution() 
        .merge()
)

# latestSeatmap

In [17]:
target_fqn = f'{catalog}.scratchpad.latestSeatmaps'

latest_seatmaps = (
    seatReservationHistory_new
    .withColumn('rn', f.row_number().over(Window.partitionBy('showtimeId').orderBy(f.col('snapshot_ts').desc())))
    .filter('rn == 1')
)

(
    latest_seatmaps
    .write
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(target_fqn)
)